In [1]:

# Download the original UCI dataset
!wget -q --show-progress "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip" -O /content/uci_electricity.zip

# Extract dataset
!unzip -o -q /content/uci_electricity.zip -d /content/uci_electricity

# Load dataset
import pandas as pd

df = pd.read_csv(
    "/content/uci_electricity/LD2011_2014.txt",
    sep=";",
    decimal=",",
    index_col=0,
    parse_dates=True
)

print("Shape:", df.shape)
print("Start:", df.index.min())
print("End:", df.index.max())
print("Consumers:", len(df.columns))


/content/uci_electr     [           <=>      ] 249.23M  53.1MB/s    in 4.8s    
Shape: (140256, 370)
Start: 2011-01-01 00:15:00
End: 2015-01-01 00:00:00
Consumers: 370


In [2]:

# STEP 1: DATA QUALITY CHECK

import pandas as pd
import numpy as np

print("Dataset shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate timestamps:", int(df.index.duplicated().sum()))

# Check active consumers
nonzero_counts = df.fillna(0).ne(0).sum()

print("\nCompletely zero-valued consumers:",
      int((nonzero_counts == 0).sum()))

print("\nTop 5 consumers with most nonzero readings:")
print(nonzero_counts.sort_values(ascending=False).head())

# Check the first consumer
print("\nMT_001 nonzero observations:",
      int(df["MT_001"].fillna(0).ne(0).sum()))

# Check if 2014 observations exist
print("\n2014 observations:",
      df.loc["2014"].shape[0])


Dataset shape: (140256, 370)
Missing values: 0
Duplicate timestamps: 0

Completely zero-valued consumers: 0

Top 5 consumers with most nonzero readings:
MT_330    140256
MT_329    140256
MT_326    140256
MT_327    140256
MT_324    140256
dtype: int64

MT_001 nonzero observations: 99513

2014 observations: 35040


In [3]:

# STEP 2: TARGET ANALYSIS

import pandas as pd
import numpy as np

# Two possible forecasting targets
targets = {
    "MT_001": df["MT_001"].astype(float),
    "Total_Load": df.sum(axis=1).astype(float)
}

for name, series in targets.items():
    print("\n" + "=" * 50)
    print("TARGET:", name)
    print("=" * 50)

    print("Mean:", round(series.mean(), 2))
    print("Median:", round(series.median(), 2))
    print("Maximum:", round(series.max(), 2))
    print("Zero values:", int((series == 0).sum()))

    # Analyze values by year
    yearly = pd.DataFrame({
        "Count": series.groupby(series.index.year).size(),
        "Mean": series.groupby(series.index.year).mean(),
        "Nonzero %": (
            series.ne(0).groupby(series.index.year).mean() * 100
        )
    })

    print("\nYearly statistics:")
    print(yearly.round(2))



TARGET: MT_001
Mean: 3.97
Median: 1.27
Maximum: 48.22
Zero values: 40743

Yearly statistics:
      Count  Mean  Nonzero %
2011  35039  0.00       0.00
2012  35136  5.50      99.96
2013  35040  6.32      94.71
2014  35040  4.06      89.05
2015      1  2.54     100.00

TARGET: Total_Load
Mean: 195556.94
Median: 167856.51
Maximum: 452652.6
Zero values: 0

Yearly statistics:
      Count       Mean  Nonzero %
2011  35039  122716.73      100.0
2012  35136  212317.43      100.0
2013  35040  222736.77      100.0
2014  35040  224410.87      100.0
2015      1  124214.20      100.0


In [4]:

# STEP 3: HOURLY AGGREGATION + BASELINE EVALUATION

import pandas as pd
import numpy as np

# Sum the readings from 370 consumers
total_load = df.sum(axis=1).astype(float)

# Treat timestamps as 15-minute interval end times.
# Shift by 15 minutes for complete hourly grouping.
total_load.index = total_load.index - pd.Timedelta(minutes=15)

# Aggregate to hourly mean values
hourly_load = total_load.resample("1h").mean()
hourly_load.name = "total_load"

print("Hourly observations:", len(hourly_load))
print("Start:", hourly_load.index.min())
print("End:", hourly_load.index.max())
print("Missing:", hourly_load.isna().sum())

# Fixed chronological evaluation:
# Train: 2011-2012
# Validation: 2013
# Test: 2014
train = hourly_load.loc["2011":"2012"]
val = hourly_load.loc["2013"]
test = hourly_load.loc["2014"]

print("\nTrain hours:", len(train))
print("Validation hours:", len(val))
print("Test hours:", len(test))

# Seasonal naive baselines
# Predict from the same hour yesterday / last week
naive24 = hourly_load.shift(24).loc[test.index]
naive168 = hourly_load.shift(168).loc[test.index]

def evaluate(y_true, y_pred):
    valid = y_true.notna() & y_pred.notna()
    actual = y_true.loc[valid].to_numpy()
    predicted = y_pred.loc[valid].to_numpy()

    rmse = np.sqrt(np.mean((actual - predicted)**2))

    nonzero = np.abs(actual) > 1e-8
    mape = np.mean(
        np.abs((actual[nonzero] - predicted[nonzero])
               / actual[nonzero])
    ) * 100

    return rmse, mape

for name, predictions in [
    ("Naive-24", naive24),
    ("Naive-168", naive168)
]:
    rmse, mape = evaluate(test, predictions)
    print(f"{name}: RMSE={rmse:.2f}, MAPE={mape:.2f}%")


Hourly observations: 35064
Start: 2011-01-01 00:00:00
End: 2014-12-31 23:00:00
Missing: 0

Train hours: 17544
Validation hours: 8760
Test hours: 8760
Naive-24: RMSE=13555.31, MAPE=3.68%
Naive-168: RMSE=20769.35, MAPE=5.13%


In [5]:

# STEP 4: LEAKAGE-SAFE XGBOOST
# One-step-ahead electricity load forecasting

import pandas as pd
import numpy as np
from xgboost import XGBRegressor

# Hourly total load from the previous step
s = hourly_load.astype(float).sort_index()

data = pd.DataFrame({"target": s})

# Lag features: only previous observations
for lag in [1, 24, 48, 168]:
    data[f"lag_{lag}"] = s.shift(lag)

# Rolling statistics from PAST values only
past = s.shift(1)

data["rolling_mean_24"] = past.rolling(24).mean()
data["rolling_mean_168"] = past.rolling(168).mean()
data["rolling_std_24"] = past.rolling(24).std()

# Calendar features
data["hour"] = data.index.hour
data["day_of_week"] = data.index.dayofweek
data["month"] = data.index.month
data["is_weekend"] = (data.index.dayofweek >= 5).astype(int)

# Remove early rows without sufficient history
data = data.dropna()

features = data.drop(columns=["target"])
target = data["target"]

# Chronological split
train_mask = data.index.year <= 2012
val_mask = data.index.year == 2013

X_train = features.loc[train_mask]
y_train = target.loc[train_mask]

X_val = features.loc[val_mask]
y_val = target.loc[val_mask]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Number of features:", X_train.shape[1])

# Fixed initial model configuration
model = XGBRegressor(
    n_estimators=350,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=2
)

model.fit(X_train, y_train)

# Evaluate on validation data only
pred_val = model.predict(X_val)

rmse = np.sqrt(
    np.mean((y_val.to_numpy() - pred_val) ** 2)
)

mape = np.mean(
    np.abs(
        (y_val.to_numpy() - pred_val)
        / y_val.to_numpy()
    )
) * 100

print("\nXGBOOST VALIDATION RESULTS")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

# Comparable naive baselines on 2013
for lag in [24, 168]:
    naive_pred = s.shift(lag).loc[y_val.index]
    errors = y_val - naive_pred

    naive_rmse = np.sqrt(np.mean(errors ** 2))
    naive_mape = np.mean(
        np.abs(errors / y_val)
    ) * 100

    print(
        f"Naive-{lag}: "
        f"RMSE={naive_rmse:.2f}, "
        f"MAPE={naive_mape:.2f}%"
    )


Training samples: 17376
Validation samples: 8760
Number of features: 11

XGBOOST VALIDATION RESULTS
RMSE: 7195.67
MAPE: 2.31%
Naive-24: RMSE=13459.18, MAPE=3.84%
Naive-168: RMSE=19947.32, MAPE=5.24%


In [6]:

# STEP 5: XGBOOST FINAL TEST
# Keep the hyperparameters selected before test evaluation

import numpy as np
import pandas as pd
from sklearn.base import clone

# Use data and features from STEP 4
train_full_mask = data.index.year <= 2013
test_mask = data.index.year == 2014

X_train_full = features.loc[train_full_mask]
y_train_full = target.loc[train_full_mask]

X_test = features.loc[test_mask]
y_test = target.loc[test_mask]

# Create a fresh model with the SAME parameters
final_model = clone(model)

# Train on 2011-2013
final_model.fit(X_train_full, y_train_full)

# Predict on 2014
pred_test = final_model.predict(X_test)

def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    rmse = np.sqrt(np.mean((actual - predicted) ** 2))

    nonzero = np.abs(actual) > 1e-8
    mape = np.mean(
        np.abs(
            (actual[nonzero] - predicted[nonzero])
            / actual[nonzero]
        )
    ) * 100

    return rmse, mape

print("TRAIN SAMPLES:", len(X_train_full))
print("TEST SAMPLES:", len(X_test))

print("\n=== FINAL TEST RESULTS (2014) ===")

results = []

# XGBoost
rmse, mape = calculate_metrics(y_test, pred_test)
results.append(["XGBoost", rmse, mape])

# Naive models: previous actual values are available
for lag in [24, 168]:
    naive_pred = s.shift(lag).loc[y_test.index]

    rmse, mape = calculate_metrics(
        y_test,
        naive_pred
    )
    results.append([f"Naive-{lag}", rmse, mape])

results_df = pd.DataFrame(
    results,
    columns=["Model", "RMSE", "MAPE (%)"]
)

print(results_df.round(2).to_string(index=False))


TRAIN SAMPLES: 26136
TEST SAMPLES: 8760

=== FINAL TEST RESULTS (2014) ===
    Model     RMSE  MAPE (%)
  XGBoost  5906.86      1.91
 Naive-24 13555.31      3.68
Naive-168 20769.35      5.13


In [7]:

# STEP 6: LSTM VALIDATION
# UCI Electricity Load Diagrams 2011-2014
# Target: hourly total load
# Forecast: one hour ahead

import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

# Use hourly data from previous steps
series = hourly_load.astype("float32").sort_index()

SEQ_LEN = 168

# Fit scaler ONLY on 2011-2012 training data
train_series = series.loc["2011":"2012"]

scaler = StandardScaler()
scaler.fit(train_series.to_numpy().reshape(-1, 1))

scaled = scaler.transform(
    series.to_numpy().reshape(-1, 1)
).ravel().astype("float32")

# Build sequences using PAST observations only
X = np.array([
    scaled[i - SEQ_LEN:i]
    for i in range(SEQ_LEN, len(scaled))
], dtype=np.float32).reshape(-1, SEQ_LEN, 1)

y = scaled[SEQ_LEN:]
timestamps = series.index[SEQ_LEN:]

# Chronological split
train_mask = timestamps.year <= 2012
val_mask = timestamps.year == 2013

X_train_lstm = X[train_mask]
y_train_lstm = y[train_mask]

X_val_lstm = X[val_mask]
y_val_lstm = y[val_mask]

print("Training:", X_train_lstm.shape)
print("Validation:", X_val_lstm.shape)

# LSTM architecture
lstm_model = Sequential([
    Input(shape=(SEQ_LEN, 1)),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1)
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse"
)

# Stop training when validation no longer improves
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train_lstm,
    y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    epochs=20,
    batch_size=128,
    callbacks=[early_stop],
    shuffle=False,
    verbose=1
)

# Predict validation period
pred_scaled = lstm_model.predict(
    X_val_lstm, verbose=0
)

pred_val_lstm = scaler.inverse_transform(
    pred_scaled
).ravel()

actual_val_lstm = series.loc[
    timestamps[val_mask]
].to_numpy()

# Evaluation
rmse_lstm = np.sqrt(
    np.mean((actual_val_lstm - pred_val_lstm) ** 2)
)

mape_lstm = np.mean(
    np.abs(
        (actual_val_lstm - pred_val_lstm)
        / actual_val_lstm
    )
) * 100

best_epoch = np.argmin(
    history.history["val_loss"]
) + 1

print("\n=== LSTM VALIDATION (2013) ===")
print("Best epoch:", best_epoch)
print(f"RMSE: {rmse_lstm:.2f}")
print(f"MAPE: {mape_lstm:.2f}%")


Training: (17376, 168, 1)
Validation: (8760, 168, 1)
Epoch 1/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 15s 94ms/step - loss: 0.2768 - val_loss: 0.3394
Epoch 2/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 12s 92ms/step - loss: 0.0856 - val_loss: 0.4981
Epoch 3/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - loss: 0.0768 - val_loss: 0.4315
Epoch 4/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 20s 95ms/step - loss: 0.0628 - val_loss: 0.4248
Epoch 5/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 20s 92ms/step - loss: 0.0572 - val_loss: 0.3303
Epoch 6/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 20s 87ms/step - loss: 0.0466 - val_loss: 0.2899
Epoch 7/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 13s 98ms/step - loss: 0.0392 - val_loss: 0.2310
Epoch 8/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - loss: 0.0321 - val_loss: 0.1888
Epoch 9/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 20s 91ms/step - loss: 0.0273 - val_loss: 0.1747
Epoch 10/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 21s 97ms/step - loss: 0.0250 - val_loss: 0.1787
Epoch 11/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 20s 93ms/step - loss: 0.0

In [8]:

# STEP 7: CONTINUE LSTM TRAINING
# Uses the existing model, data and scaler

import copy
import numpy as np
import tensorflow as tf

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

# Keep the original 20-epoch model
old_weights = copy.deepcopy(lstm_model.get_weights())
old_best_loss = min(history.history["val_loss"])

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )
]

# Continue from epoch 20, up to epoch 50
extra_history = lstm_model.fit(
    X_train_lstm,
    y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    initial_epoch=20,
    epochs=50,
    batch_size=128,
    shuffle=False,
    callbacks=callbacks,
    verbose=2
)

new_losses = extra_history.history["val_loss"]
new_best_loss = min(new_losses)

if new_best_loss >= old_best_loss:
    lstm_model.set_weights(old_weights)
    chosen_epoch = 20
    best_loss = old_best_loss
else:
    chosen_epoch = 20 + int(np.argmin(new_losses)) + 1
    best_loss = new_best_loss

# Evaluate the selected model on 2013 validation
pred_scaled = lstm_model.predict(
    X_val_lstm, verbose=0
)

pred_val_lstm = scaler.inverse_transform(
    pred_scaled
).ravel()

actual = actual_val_lstm.astype(float)

rmse = np.sqrt(
    np.mean((actual - pred_val_lstm) ** 2)
)

mape = np.mean(
    np.abs((actual - pred_val_lstm) / actual)
) * 100

print("\n=== UPDATED LSTM VALIDATION ===")
print("Selected epoch:", chosen_epoch)
print(f"Best val_loss: {best_loss:.4f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")


Epoch 21/50
136/136 - 16s - 121ms/step - loss: 0.0157 - val_loss: 0.1208 - learning_rate: 0.0010
Epoch 22/50
136/136 - 15s - 110ms/step - loss: 0.0154 - val_loss: 0.1149 - learning_rate: 0.0010
Epoch 23/50
136/136 - 22s - 165ms/step - loss: 0.0153 - val_loss: 0.1105 - learning_rate: 0.0010
Epoch 24/50
136/136 - 19s - 142ms/step - loss: 0.0151 - val_loss: 0.1077 - learning_rate: 0.0010
Epoch 25/50
136/136 - 21s - 153ms/step - loss: 0.0151 - val_loss: 0.1043 - learning_rate: 0.0010
Epoch 26/50
136/136 - 13s - 95ms/step - loss: 0.0149 - val_loss: 0.1023 - learning_rate: 0.0010
Epoch 27/50
136/136 - 12s - 89ms/step - loss: 0.0148 - val_loss: 0.1010 - learning_rate: 0.0010
Epoch 28/50
136/136 - 12s - 88ms/step - loss: 0.0147 - val_loss: 0.0999 - learning_rate: 0.0010
Epoch 29/50
136/136 - 13s - 99ms/step - loss: 0.0145 - val_loss: 0.1001 - learning_rate: 0.0010
Epoch 30/50
136/136 - 20s - 144ms/step - loss: 0.0144 - val_loss: 0.0980 - learning_rate: 0.0010
Epoch 31/50
136/136 - 13s - 93ms/s

In [9]:

# STEP 8: FINAL LSTM TEST
# UCI Electricity Load Diagrams 2011-2014
# Rolling one-step-ahead forecast

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.callbacks import LearningRateScheduler

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

series = hourly_load.astype("float32").sort_index()
SEQ_LEN = 168

# 1. Fit scaler on 2011-2013 only
training_series = series.loc["2011":"2013"]

final_scaler = StandardScaler()
final_scaler.fit(
    training_series.to_numpy().reshape(-1, 1)
)

scaled_all = final_scaler.transform(
    series.to_numpy().reshape(-1, 1)
).ravel().astype("float32")

# 2. Create lagged sequences
X_all = np.array(
    [
        scaled_all[i - SEQ_LEN:i]
        for i in range(SEQ_LEN, len(scaled_all))
    ],
    dtype=np.float32
).reshape(-1, SEQ_LEN, 1)

y_all = scaled_all[SEQ_LEN:]
timestamps = series.index[SEQ_LEN:]

train_mask = timestamps.year <= 2013
test_mask = timestamps.year == 2014

X_train_final = X_all[train_mask]
y_train_final = y_all[train_mask]

X_test_final = X_all[test_mask]
y_test_actual = series.loc[
    timestamps[test_mask]
].to_numpy()

print("Training:", X_train_final.shape)
print("Test:", X_test_final.shape)

# 3. Same architecture as the validation model
final_lstm = Sequential([
    Input(shape=(SEQ_LEN, 1)),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1)
])

final_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse"
)

# 4. Fixed learning-rate schedule selected
# from the 2013 validation experiment
def fixed_lr(epoch, current_lr):
    if epoch < 33:
        return 0.001
    elif epoch < 41:
        return 0.0005
    elif epoch < 45:
        return 0.00025
    elif epoch < 48:
        return 0.000125
    else:
        return 0.0000625

# 5. Train from scratch for 49 epochs
final_history = final_lstm.fit(
    X_train_final,
    y_train_final,
    epochs=49,
    batch_size=128,
    shuffle=False,
    callbacks=[
        LearningRateScheduler(fixed_lr)
    ],
    verbose=2
)

# 6. Evaluate ONCE on the 2014 test set
pred_scaled = final_lstm.predict(
    X_test_final,
    verbose=0
)

pred_lstm_test = final_scaler.inverse_transform(
    pred_scaled
).ravel()

actual = y_test_actual.astype(float)
predicted = pred_lstm_test.astype(float)

lstm_rmse = np.sqrt(
    np.mean((actual - predicted) ** 2)
)

lstm_mape = np.mean(
    np.abs((actual - predicted) / actual)
) * 100

print("\n=== FINAL LSTM TEST (2014) ===")
print(f"RMSE: {lstm_rmse:.2f}")
print(f"MAPE: {lstm_mape:.2f}%")

print("\n=== XGBOOST VS LSTM (2014) ===")
print("XGBoost: RMSE=5906.86, MAPE=1.91%")
print(
    f"LSTM:    RMSE={lstm_rmse:.2f}, "
    f"MAPE={lstm_mape:.2f}%"
)


Training: (26136, 168, 1)
Test: (8760, 168, 1)
Epoch 1/49
205/205 - 18s - 90ms/step - loss: 0.2408 - learning_rate: 0.0010
Epoch 2/49
205/205 - 20s - 98ms/step - loss: 0.0789 - learning_rate: 0.0010
Epoch 3/49
205/205 - 21s - 101ms/step - loss: 0.0734 - learning_rate: 0.0010
Epoch 4/49
205/205 - 20s - 99ms/step - loss: 0.0481 - learning_rate: 0.0010
Epoch 5/49
205/205 - 20s - 100ms/step - loss: 0.0376 - learning_rate: 0.0010
Epoch 6/49
205/205 - 14s - 70ms/step - loss: 0.0257 - learning_rate: 0.0010
Epoch 7/49
205/205 - 14s - 68ms/step - loss: 0.0226 - learning_rate: 0.0010
Epoch 8/49
205/205 - 14s - 69ms/step - loss: 0.0209 - learning_rate: 0.0010
Epoch 9/49
205/205 - 14s - 68ms/step - loss: 0.0200 - learning_rate: 0.0010
Epoch 10/49
205/205 - 14s - 69ms/step - loss: 0.0196 - learning_rate: 0.0010
Epoch 11/49
205/205 - 14s - 69ms/step - loss: 0.0196 - learning_rate: 0.0010
Epoch 12/49
205/205 - 14s - 69ms/step - loss: 0.0186 - learning_rate: 0.0010
Epoch 13/49
205/205 - 21s - 102ms/st